# 02 — Source-level outer and inner splits

Splits are assigned on a table with exactly one row per source video and
expanded to sequences afterward. Every clip, mirror, and augmentation from a
video inherits that video's fold. Outer-test sources are never used for
encoder training, checkpoint selection, read-out tuning, or preprocessing
statistics. Inner folds tune read-outs using outer-training sources only.

Dataset annotations keep source counts reasonably balanced across folds and
later support one explicitly named confounding control; they are not diagnoses
or primary prediction targets. Multiple videos
may still depict the same unidentified person, so these folds establish
held-out-video—not held-out-person—evaluation.

## What experiment-design problem does this notebook solve?

A **split** assigns observations to roles before model fitting. The dangerous
shortcut would be to split 625 sequences independently. Sequences cut from the
same source video share scene, camera, clothing, pose-extraction behavior, and
often adjacent motion. Putting sibling sequences in both training and testing
would let the model encounter source-specific information before evaluation.
This is called **data leakage**.

Notebook 02 prevents that problem by collapsing the cohort to 93 source videos,
assigning folds on that source table, and only then expanding assignments back
to sequences. It creates two nested levels:

- The five **outer folds** estimate held-out-source performance. One fold is
  sealed as the final test set while the other four supply training data.
- Four **inner folds** divide only the current outer-training sources. Notebook
  04 uses them to select the ridge regularization strength of the linear
  read-out. They never open the outer-test sources and do not select or stop the
  encoder trained in notebook 03.

**Stratification** uses dataset annotations to keep source counts as even as the
available integer counts allow. It does not train a diagnosis and does not
guarantee equal sequence counts, because sources yield different numbers of
sequences. The split seed `20260904` makes the assignment reproducible; it is
separate from optimization seeds 42–46 used during model training.

<svg viewBox="0 0 1040 330" width="100%" role="img"
     aria-labelledby="split-flow-title split-flow-description"
     xmlns="http://www.w3.org/2000/svg">
  <title id="split-flow-title">Nested source-video splitting workflow</title>
  <desc id="split-flow-description">Ninety-three source videos are assigned to
  five outer folds. For one outer fold, 18 or 19 test sources remain sealed and
  74 or 75 training sources are subdivided into four inner read-out folds. All
  sequences inherit the role of their source video.</desc>
  <defs><marker id="arrow02" markerWidth="8" markerHeight="8" refX="7"
    refY="4" orient="auto"><path d="M0,0 L8,4 L0,8 z" fill="#475569"/></marker></defs>
  <style>
    .box02{fill:#f8fafc;stroke:#334155;stroke-width:1.5}
    .train02{fill:#ecfdf5;stroke:#047857;stroke-width:1.5}
    .test02{fill:#fff7ed;stroke:#c2410c;stroke-width:1.7}
    .inner02{fill:#eff6ff;stroke:#2563eb;stroke-width:1.5}
    .line02{stroke:#475569;stroke-width:1.8;fill:none;marker-end:url(#arrow02)}
    .h02{font:600 14px system-ui,sans-serif;fill:#0f172a}
    .s02{font:12px system-ui,sans-serif;fill:#475569}
  </style>
  <rect class="box02" x="15" y="115" width="175" height="88" rx="9"/>
  <text class="h02" x="102" y="145" text-anchor="middle">Locked cohort</text>
  <text class="s02" x="102" y="168" text-anchor="middle">93 source videos</text>
  <text class="s02" x="102" y="188" text-anchor="middle">625 sequences inherit source</text>
  <rect class="box02" x="240" y="115" width="180" height="88" rx="9"/>
  <text class="h02" x="330" y="145" text-anchor="middle">Five outer folds</text>
  <text class="s02" x="330" y="168" text-anchor="middle">stratified by annotation</text>
  <text class="s02" x="330" y="188" text-anchor="middle">each source tests once</text>
  <path class="line02" d="M190 159 L240 159"/>
  <rect class="train02" x="480" y="45" width="205" height="88" rx="9"/>
  <text class="h02" x="582" y="75" text-anchor="middle">Outer-training sources</text>
  <text class="s02" x="582" y="98" text-anchor="middle">74 or 75 sources</text>
  <text class="s02" x="582" y="118" text-anchor="middle">encoder may use these</text>
  <rect class="test02" x="480" y="220" width="205" height="88" rx="9"/>
  <text class="h02" x="582" y="250" text-anchor="middle">Outer-test sources</text>
  <text class="s02" x="582" y="273" text-anchor="middle">18 or 19 sources</text>
  <text class="s02" x="582" y="293" text-anchor="middle">sealed until final evaluation</text>
  <path class="line02" d="M420 145 C450 145 450 89 480 89"/>
  <path class="line02" d="M420 175 C450 175 450 264 480 264"/>
  <rect class="inner02" x="755" y="25" width="260" height="145" rx="9"/>
  <text class="h02" x="885" y="55" text-anchor="middle">Four inner read-out folds</text>
  <text class="s02" x="885" y="79" text-anchor="middle">inner train: 55–57 sources</text>
  <text class="s02" x="885" y="99" text-anchor="middle">inner validation: 18–19 sources</text>
  <text class="s02" x="885" y="124" text-anchor="middle">select ridge penalty only</text>
  <text class="s02" x="885" y="146" text-anchor="middle">outer test remains untouched</text>
  <path class="line02" d="M685 89 L755 89"/>
  <rect class="test02" x="755" y="220" width="260" height="88" rx="9"/>
  <text class="h02" x="885" y="250" text-anchor="middle">Final held-out evaluation</text>
  <text class="s02" x="885" y="273" text-anchor="middle">one outer fold at a time</text>
  <text class="s02" x="885" y="293" text-anchor="middle">performed in notebook 04</text>
  <path class="line02" d="M685 264 L755 264"/>
</svg>

## How to read the split-construction progress display

Four short stages load the locked cohort, assign nested source-video folds, save
the lineage-bound split manifest, and render the source-count summary. The work
is deterministic and normally completes in seconds. A full bar confirms that the
stages ran; the printed fold census and non-overlap assertions remain the evidence
that the split itself is complete and leakage-controlled.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from IPython import get_ipython
from IPython.display import display


def locate_suite_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for ancestor in (start, *start.parents):
        for candidate in (ancestor, ancestor / "neurips-laterality"):
            if (
                (candidate / "config" / "protocol.json").is_file()
                and (candidate / "laterality").is_dir()
            ):
                return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate neurips-laterality from the current working directory."
    )


SUITE_ROOT = locate_suite_root()
if str(SUITE_ROOT) not in sys.path:
    sys.path.insert(0, str(SUITE_ROOT))

from laterality.config import load_context

context = load_context(SUITE_ROOT / "config" / "protocol.json")
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")


def show_inline(figure):
    display(figure)
    plt.close(figure)


print(
    f"suite={SUITE_ROOT} profile={context.profile} "
    f"artifacts={context.artifact_root} protocol={context.protocol_digest[:12]}"
)

In [ ]:
from laterality.data import load_cohort
from laterality.splitting import build_source_splits, save_splits
from laterality.visualization import split_figure
from notebook_progress import NotebookTaskProgress

split_progress = NotebookTaskProgress(
    "Source-level split construction progress",
    "stage",
    refresh_seconds=0.25,
)
split_progress.start(4, profile=context.profile)

with split_progress.unit(1, "Load and validate the locked cohort"):
    cohort = load_cohort(context)

with split_progress.unit(2, "Assign nested folds on the source-video table"):
    split_config = context.protocol["splits"]
    splits = build_source_splits(
        cohort.table,
        context.protocol["data"]["conditions"],
        outer_folds=split_config["outer_folds"],
        inner_folds=split_config["inner_folds"],
        seed=split_config["seed"],
    )

with split_progress.unit(3, "Validate lineage and save the split manifest"):
    split_artifact = save_splits(context, cohort, splits)

with split_progress.unit(4, "Summarize and render source-count balance"):
    fold_summary = [
        {
            "fold": fold["fold"],
            "train_sources": len(fold["train_sources"]),
            "test_sources": len(fold["test_sources"]),
            "train_source_counts": fold["train_source_counts"],
            "test_source_counts": fold["test_source_counts"],
            "inner_folds": len(fold["inner_readout_folds"]),
        }
        for fold in splits["folds"]
    ]
    show_inline(split_figure(context, splits))
split_progress.complete(status="Source-level split construction complete")
{
    "split_artifact": str(split_artifact),
    "source_census": splits["source_census"],
    "folds": fold_summary,
}

## Step-by-step interpretation of the split results

### 1. Check the source census before the bars

The split contains all 93 accepted source videos and all 625 accepted
sequences from notebook 01. The source census is 9 `cerebralpalsy`, 28
`myopathic`, 29 `normal`, 9 `parkinsons`, and 18 `stroke` source videos. These
strings are dataset annotations used for stratification; they are not diagnoses
made by this project and are not the primary target.

With five folds, an annotation represented by 9 sources can contribute only one
or two sources to each test fold. Counts of 28, 29, and 18 similarly imply test
allocations of roughly 5–6, 5–6, and 3–4. The plotted bars show exactly these
integer patterns. Small differences of one are expected and are the closest
possible balance; identical fold bars are not mathematically available for all
five annotations.

### 2. Read each bar as held-out source videos, not sequences

The current paper split is:

| Outer fold | Train sources | Test sources | Train sequences | Test sequences | Test annotation counts (CP / M / N / P / S) |
|---:|---:|---:|---:|---:|---|
| 0 | 74 | 19 | 436 | 189 | 2 / 5 / 6 / 2 / 4 |
| 1 | 74 | 19 | 443 | 182 | 2 / 5 / 6 / 2 / 4 |
| 2 | 74 | 19 | 553 | 72 | 2 / 6 / 6 / 1 / 4 |
| 3 | 75 | 18 | 548 | 77 | 1 / 6 / 6 / 2 / 3 |
| 4 | 75 | 18 | 520 | 105 | 2 / 6 / 5 / 2 / 3 |

Here CP abbreviates `cerebralpalsy`, M `myopathic`, N `normal`, P
`parkinsons`, and S `stroke`. The abbreviations are only table space-savers.

Notice that folds 0 and 2 both test 19 sources but contain 189 versus 72 test
sequences. That is not a splitting failure. It shows why sequence-level
splitting would be misleading: some source videos yield many more sequences
than others. Later metrics give each source equal total weight so fold 0 does
not receive extra influence merely for containing more clips.

The five test-source sets are disjoint and collectively contain every one of
the 93 sources exactly once. Their test-sequence counts also sum to 625. For any
chosen fold, its training and test source sets do not overlap. Mirrors and
augmentations inherit the source assignment and therefore cannot cross the
boundary later.

### 3. Interpret the inner folds as tuning partitions only

Every outer-training set is split four ways for read-out tuning. When the outer
training set has 74 sources, an inner fit uses 55 or 56 sources and validates on
18 or 19. When it has 75 sources, an inner fit uses 56 or 57 and validates on 18
or 19. Each outer-training source serves as inner validation exactly once.

Because sources have unequal numbers of sequences, the corresponding inner
sequence counts range from 266 to 450 for fitting and from 80 to 197 for
validation. Again, this variability is expected. The inner validation scores
are source-weighted in notebook 04. Most importantly, none of these inner folds
contains the current outer-test sources.

A concrete example helps. In outer fold 0, 19 sources and their 189 sequences
are placed behind the final-test boundary. The encoder in notebook 03 uses only
the remaining 74 sources. Later, ridge candidates are compared using four
rotations inside those 74 sources. Only after a penalty is selected is the
read-out refit on all 74 and evaluated on the untouched 19.

### 4. Understand what the artifact proves—and what it cannot prove

The saved split manifest carries the cohort, context, protocol, and split
digests. Later stages reject it if those fingerprints do not match. This proves
deterministic bookkeeping and source-level non-overlap under the available
`video_id` field.

It cannot prove held-out-person generalization. Two source videos could depict
the same unidentified person, and this project deliberately does not infer
identity. It also cannot show that a model performs well; no encoder or read-out
score appears in this notebook.

The narrow conclusion is: **the accepted cohort has been partitioned into
complete, non-overlapping, annotation-stratified outer source folds, with inner
tuning folds confined to each outer-training set.** This is the leakage-control
foundation required by notebooks 03–05.